# Textual Stats

## Artemis: Text -- Emotions

In [ ]:
import nltk
import numpy
import pathlib
import pandas
import string

from tqdm import tqdm
from collections import Counter
from sklearn.feature_extraction.text import TfidfVectorizer

from typing import Any, Dict, Iterable, List, Optional, Union

# ROOT = pathlib.Path.cwd().parent
# artemis = pandas.read_csv(
#     ROOT / "DATA" / "artemis" / "artemis_dataset_release_v0.csv"
# )
ROOT = pathlib.Path('/mnt/HDD/DATA')
artemis = pandas.read_csv(ROOT / "ARTEMIS" / "artemis_official_data" /
                          "official_data" / "artemis_dataset_release_v0.csv")
artemis['utterance'] = artemis['utterance'].apply(str.lower)
artemis['utterance'] = artemis['utterance'].apply(
    lambda s: s.translate(str.maketrans('', '', string.punctuation))
)
artemis

### Characterization of language per emotion

In [ ]:
def tf(documents: Union[str, List[str], pandas.Series],
	   top_k: Optional[int] = None,
	   exclude: List = None) -> Dict[str, str]:
	""" Estimate the frequency of the terms in a set of documents. Words are
	 obtained splitting sentences by whitespaces.

	 Args:
	 	documents: Texts to consider.
	 	top_k: Top terms most represented.
	 	exclude: Words to exclude from count.

	Returns:
		words and their absolute frequency.
	"""
	if not isinstance(documents, str):
		documents = ' '.join(documents)
	words = documents.split(' ')
	if exclude is not None:
		words = [w for w in words if w not in exclude]
	return {x[0]: x[1] for x in Counter(words).most_common(top_k)}

def tf_idf(documents: Dict[int, str],
		   top_k: int,
		   exclude: Union[str, List] = 'english',
		   ) -> Dict[int, List[Union[list, Any]]]:
	"""
	Retrieve the top_k most characteristic words in every document according to
	a TF-IDF measure.

	Args:
		documents (Dict[int, str]):
		top_k (int): Number of relevant words per document.
		exclude: Terms to exclude from analysis.

	Returns:
		Dict[int, List[Union[list, Any]]]
	"""
	docs = [' '.join(val) for key, val in documents.items()]
	keys = list(documents.keys())
	vectorizer = TfidfVectorizer(strip_accents='unicode',
								 lowercase=True,
								 stop_words=exclude)
	vec_docs = vectorizer.fit_transform(docs)
	feature_names = vectorizer.get_feature_names_out()
	relevant_words = dict()
	for i in tqdm(range(vec_docs.shape[0])):
		row = vec_docs.getrow(i).toarray()[0]
		k_words = numpy.flip(list(numpy.argsort(row)[-top_k:]))
		relevant_words[keys[i]] = [feature_names[j] for j in k_words]
	return relevant_words


stopwords = nltk.corpus.stopwords.words('english') + [''] + ['like', 'feel',
															 'painting',
															 'look',
															 'looks',
															 'looking',
															 'makes',
															 'colors']

emotion_labels = artemis['emotion'].unique()
utts_by_emotion = dict()
for l, label in enumerate(emotion_labels):
	artemis_l = artemis[artemis['emotion'] == label]
	utts_by_emotion[label] = artemis_l['utterance'].values
	print(f"Label: {label}")
	print(f"  > {len(artemis_l) / len(artemis):.2f}% ({len(artemis_l)})")
	most_freq_words = tf(documents=artemis_l['utterance'], top_k=10,
						 exclude=stopwords)
	print(f"  > Most common words: {most_freq_words}")
	print('-' * 79)

most_rep_words = tf_idf(documents=utts_by_emotion, top_k=10, exclude=stopwords)
print(f"  > Most representative words per emotion: {most_rep_words}")


## Artemis: Text -> Style

In [ ]:
style_labels = artemis['art_style'].unique()
utts_by_style = dict()
for l, label in enumerate(style_labels):
	artemis_l = artemis[artemis['art_style'] == label]
	utts_by_style[label] = artemis_l['utterance'].values
	print(f"Label: {label}")
	print(f"  > {len(artemis_l) / len(artemis):.2f}% ({len(artemis_l)})")
	most_freq_words = tf(documents=artemis_l['utterance'], top_k=5,
						 exclude=stopwords)
	print(f"  > Most common words: {most_freq_words}")
	print('-' * 79)

most_rep_words = tf_idf(documents=utts_by_style, top_k=5, exclude=stopwords)

print("Most representative words per style:")
for style in style_labels:
	print(f"  > Style = {style}")
	print(f"    {most_rep_words[style]}")

## Does text in Artpedia behave like artistic language?

In [ ]:
from textblob import TextBlob
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

class VaderAnalysis:

	def __init__(self):
		self.vader = SentimentIntensityAnalyzer()

	def __call__(self, text):
		df = {'sentence': [], 'pos': [], 'neg': [], 'neu': [], 'compound': []}
		for sentence in text:
			sentiment_dict = self.vader.polarity_scores(sentence)
			df['sentence'].append(sentence)
			for key in sentiment_dict:
				df[key].append(sentiment_dict[key])
		df = pandas.DataFrame(df)
		return pandas.DataFrame(df)


class TextBlobAnalysis:

	def __init__(self, subjectivity=True, POS=False):
		"""
		TODO
		"""
		self.textblob = TextBlob

	def __call__(self, text):
		df = {'sentence': [], 'subjectivity': []}
		for sentence in text:
			blob = self.textblob(sentence)
			df['sentence'].append(sentence)
			df['subjectivity'].append(blob.subjectivity)
		return pandas.DataFrame(df)


class BrysbaertConcreteness:

	def __init__(self, path_to_file):
		"""
		TODO
		"""
		self.df = pandas.read_csv(path_to_file, sep='\t')
		self.gram_extractor = TextBlob

	def __call__(self, text):
		df = {'sentence': [], 'concreteness': []}
		for sentence in tqdm(text, total=len(text)):
			sentence_score = []
			df['sentence'].append(sentence)

			bi_grams = self.gram_extractor(sentence).ngrams(n=2)
			for bi_gram in bi_grams:
				idx = self._find_match(bi_gram, bigram=True)
				if idx is not None:
					sentence_score.append(self.df['Conc.M'].loc[idx])
					sentence = sentence.replace(self.df['Word'].loc[idx], '')
			words = sentence.split()
			for word in words:
				idx = self._find_match(word)
				if idx is not None:
					sentence_score.append(self.df['Conc.M'].loc[idx])

			sentence_score = numpy.mean(sentence_score) if sentence_score else None
			df['concreteness'].append(sentence_score)

		return pandas.DataFrame(df)

	def _find_match(self, word, bigram=False):
		"""
		TODO
		"""
		if bigram:
			word = ' '.join(word)
		match = self.df[self.df['Word'] == word]
		if not match.empty:
			return match.index[0]
		return None


In [ ]:
artpedia_root = pathlib.Path('/mnt/HDD/DATA/Artpedia/artpedia.json')
artpedia = pandas.read_json(artpedia_root, orient='index')
artpedia

### Sentiment

In [ ]:
content_text = [s for sent in artpedia['visual_sentences'] for s in sent]

vader = VaderAnalysis()
sentiment = vader(content_text)
sentiment['compound'].hist(bins=50)

In [ ]:
context_text = [s for sent in artpedia['contextual_sentences'] for s in sent]
sentiment = vader(context_text)
sentiment['compound'].hist(bins=50)

### Subjectivity

In [ ]:
blob = TextBlobAnalysis()
subjectivity = blob(content_text)
subjectivity.hist(bins=50)

In [ ]:
blob = TextBlobAnalysis()
subjectivity = blob(context_text)
subjectivity.hist(bins=50)

### Concreteness

In [ ]:
brys = BrysbaertConcreteness('./Brysbaert_concreteness_vocabulary.csv')
concreteness = brys(content_text)
concreteness.hist(bins=50)

In [ ]:
brys = BrysbaertConcreteness('./Brysbaert_concreteness_vocabulary.csv')
concreteness = brys(context_text)
concreteness.hist(bins=50)